# 02 — Cleaning & Filtering
Loads `data/raw/variants.parquet` from notebook 01, applies quality filters via `src/filter.py`, and saves the cleaned output to `data/processed/`.

In [4]:
import sys, os
from pathlib import Path

import pandas as pd

ROOT = Path(os.getcwd()).parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from filter import full_filter_pipeline

In [5]:
# ── Configure ──────────────────────────────────────────────────────────────
IN_PATH  = ROOT / 'data' / 'raw' / 'variants.parquet'
OUT_DIR  = ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / 'variants_filtered.parquet'

# Minimum allele frequency to keep (0.001 = 0.1%)
MIN_MAF = 0.001

In [6]:
# ── Load ───────────────────────────────────────────────────────────────────
variants = pd.read_parquet(IN_PATH)
print(f'Loaded {len(variants):,} variants')
variants.head()

Loaded 1,169,063 variants


,chrom,pos,ref,alt,qual,filter_status,af_global,af_afr,af_eur,af_eas,af_amr,af_sas,ac,an,variant_type,source_file
0,chrY,2781489,C,T,None,AC0,0.0,0.0,0.0,0.0,0.0,0.0,0,2716,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
1,chrY,2781499,C,A,None,AC0,0.0,0.0,0.0,0.0,0.0,0.0,0,4097,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
2,chrY,2781511,C,G,None,AC0,0.0,0.0,0.0,0.0,0.0,0.0,0,5637,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
3,chrY,2781514,C,A,None,AC0;AS_VQSR,0.0,0.0,0.0,0.0,0.0,0.0,0,6022,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
4,chrY,2781520,G,C,None,AC0;AS_VQSR,0.0,0.0,0.0,0.0,0.0,0.0,0,6808,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz


In [7]:
# ── Filter ─────────────────────────────────────────────────────────────────
filtered = full_filter_pipeline(variants, min_maf=MIN_MAF)
filtered.head()

Filtering: 1169063 → 37089 variants (removed 1131974)


,chrom,pos,ref,alt,qual,filter_status,af_global,af_afr,af_eur,af_eas,af_amr,af_sas,ac,an,variant_type,source_file
52,chrY,2781758,C,T,None,PASS,0.002024,0.000000,0.000000,0.032258,0.006849,0.000000,3,1482,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
65,chrY,2781761,CAAAAAAAA,C,None,PASS,0.002469,0.013158,0.000000,0.000000,0.000000,0.000000,3,1215,INDEL,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
114,chrY,2782660,G,A,None,PASS,0.001632,0.001269,0.002425,0.000000,0.002979,0.000000,55,33697,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
130,chrY,2782986,C,T,None,PASS,0.001306,0.000000,0.000000,0.000000,0.000000,0.030028,43,32920,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
139,chrY,2783081,G,A,None,PASS,0.004827,0.003770,0.009193,0.000000,0.000294,0.000000,152,31491,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz


In [8]:
# ── Save ───────────────────────────────────────────────────────────────────
filtered.to_parquet(OUT_PATH, index=False, engine='pyarrow')
print(f'Saved → {OUT_PATH}')

Saved → /mnt/c/IISER/SEM_8/ds_practice/genomic-variation-landscape-and-population-comparison-main/genomic-variation-landscape-and-population-comparison-main/data/processed/variants_filtered.parquet
